<a href="https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Abstract

This project asks whether historical search-performance and content characteristics can help prioritize webpages for potential content refresh. The analysis uses the FlyRank ML Internship dataset and focuses on content-level search-performance signals available before the outcome period. A classification model was evaluated against a transparent baseline using an honest validation design and leakage checks. The measured results provide directional evidence about which webpages may be worth reviewing first, rather than evidence that a refresh will cause improved rankings or traffic. The final output is a ranked decision-support queue that can help SEO teams focus human review on higher-priority pages.

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

### Research question

Can historical search-performance and content characteristics help prioritize webpages that may need a content refresh?

### Decision supported

The analysis is designed to support the decision of **which webpages should be reviewed first** by an SEO specialist or content manager.

Instead of manually reviewing every webpage, the model provides a ranked set of pages that can be considered for review based on observed historical signals.

### Unit of analysis

The unit of analysis is **one webpage/content item**.

### Output

The model produces a **score/ranking** that can be used to prioritize webpages for human review.

### Human action

A content or SEO specialist can review the highest-priority pages and decide whether an update is appropriate based on the page's actual content, business context, and current search performance.

### Cost of a wrong call

A false positive may cause a team to spend time reviewing a page that does not require an update. A false negative may cause a page with declining performance to be reviewed later than it otherwise would have been.

### Why ML helps

A simple fixed rule may overlook interactions between multiple signals such as recent impressions, recent clicks, average position, query visibility, query concentration, and content age. The model provides a directional prioritization signal rather than a guarantee that a page will decline or that a refresh will improve performance.

In [ ]:
# Basic project framing check

print("Unit of analysis: webpage/content item")
print("Output: ranked prioritization score")
print("Decision: which webpages should be reviewed first")
print("Use: decision-support for human-reviewed content prioritization")

Unit of analysis: webpage/content item
Output: ranked prioritization score
Decision: which webpages should be reviewed first
Use: decision-support for human-reviewed content prioritization


## 2. Data

This project uses the FlyRank ML Internship data release described in the project data documentation.

The warehouse release is `v20260703`, covering search and content-performance data from **2025-01-27 through 2026-06-30**. The available warehouse tables include client, content, daily content-performance, and 90-day query-level context tables.

The project work also used the provided anonymized content-refresh dataset during model development. The starter dataset contains **30,000 rows and 44 columns**, with one row representing one pseudonymized content item.

The analysis uses historical performance information to construct features that would be available before the outcome being evaluated. The final feature set used in the validated model was:

- `previous_30d_impressions`
- `previous_30d_clicks`
- `previous_30d_avg_position`
- `visible_query_count`
- `top_query_share`
- `age_days`

Several fields were deliberately excluded. `is_declining`, `trend_direction`, and `trend_pct` were excluded because they are directly related to the target definition and could introduce label leakage. Outcome-period fields such as `march_impressions` were also not used as model features.

Pseudonymous identifiers such as client and content IDs were used for grouping and joining where required, but were not used as predictive features.

The data is treated as historical observational data. Therefore, the analysis measures associations and predictive usefulness rather than causal effects of content updates.

In [ ]:
# Define the final feature set used in the validated model

feature_cols = [
    "previous_30d_impressions",
    "previous_30d_clicks",
    "previous_30d_avg_position",
    "visible_query_count",
    "top_query_share",
    "age_days"
]

print("Final model features:")
for feature in feature_cols:
    print("-", feature)

print("\nNumber of model features:", len(feature_cols))

Final model features:
- previous_30d_impressions
- previous_30d_clicks
- previous_30d_avg_position
- visible_query_count
- top_query_share
- age_days

Number of model features: 6


## 3. Methodology

### Problem type

The project is framed as a **classification and prioritization** task. The model estimates whether a webpage belongs to the declining-performance class, and the resulting scores can be used to rank pages for review.

### Target / proxy

The target is a proxy for declining search performance derived from the available historical performance data. It is not a direct observation of whether a human editor believes that a page "needs a refresh."

This distinction is important: the model evaluates a defined performance outcome, while the final content-refresh decision remains a human decision.

### Model

The Week-5 model uses a **Random Forest classifier**. Random Forest was selected because it can model nonlinear relationships between multiple features and provides feature-importance information that can be inspected for interpretation.

### Features

The final feature set contains:

1. `previous_30d_impressions`
2. `previous_30d_clicks`
3. `previous_30d_avg_position`
4. `visible_query_count`
5. `top_query_share`
6. `age_days`

These features represent historical search performance, query visibility, query concentration, and content age.

### Baseline

The model is compared with the Week-4 rule-based baseline using the same evaluation population and metric where available.

The baseline provides a transparent reference point for determining whether the machine-learning approach provides useful additional ranking or classification signal.

### Validation

The initial random split was useful for development, but it allowed the same clients to appear in both training and testing data. The validation audit therefore introduced a more honest split grouped by client.

The grouped evaluation better reflects the question of whether the model can generalize beyond clients represented during training.

### Leakage checks

The final feature set was audited for known leakage risks.

Label-derived fields such as `is_declining`, `trend_direction`, and `trend_pct` were excluded. Outcome-period fields such as `march_impressions` were also excluded.

Pseudonymous identifiers were not used as predictive features.

The feature set therefore avoids the known label-derived and identifier leakage risks identified during the validation audit.

In [ ]:
# Final feature and leakage audit

leakage_candidates = [
    "is_declining",
    "trend_direction",
    "trend_pct",
    "march_impressions"
]

print("Final model features:")
print(feature_cols)

print("\nLeakage candidates present in features:")
found_leaks = [c for c in leakage_candidates if c in feature_cols]

if found_leaks:
    print("WARNING:", found_leaks)
else:
    print("PASS: No known label-derived/outcome fields are model features.")

id_candidates = [
    "client_hash_id",
    "content_hash_id",
    "client_id",
    "content_id"
]

found_ids = [c for c in id_candidates if c in feature_cols]

print("\nIdentifier fields present in features:")
print(found_ids if found_ids else "None")

Final model features:
['previous_30d_impressions', 'previous_30d_clicks', 'previous_30d_avg_position', 'visible_query_count', 'top_query_share', 'age_days']

Leakage candidates present in features:
PASS: No known label-derived/outcome fields are model features.

Identifier fields present in features:
None


## 4. Results (vs baseline)

The model is evaluated against the transparent baseline using the same evaluation population and metric.

The evaluation reports the positive-class base rate alongside the model results so that the measured performance is interpreted in context.

The random split used during development produced an Average Precision of **0.3874** and ROC-AUC of **0.7436**, with a positive-class base rate of approximately **19.98%**.

However, the random split had client overlap between training and testing data. Therefore, the grouped-by-client evaluation is treated as the more honest estimate of generalization.

The final comparison below will use the results produced by the executed capstone notebook rather than relying on the earlier development result.

In [ ]:
# Results receipt
# This cell should be replaced/extended with the actual final model
# and baseline metrics already produced by the capstone.

print("Random-split development result")
print("Average Precision: 0.3874")
print("ROC-AUC: 0.7436")
print("Base rate: 0.1998")
print("Client overlap: 40")

print("\nThe grouped validation result should be reported as the primary")
print("honest evaluation once reproduced in this notebook.")

Random-split development result
Average Precision: 0.3874
ROC-AUC: 0.7436
Base rate: 0.1998
Client overlap: 40

The grouped validation result should be reported as the primary
honest evaluation once reproduced in this notebook.


## 5. Limitations

This analysis has several important limitations.

First, the target is a proxy for declining search performance rather than a direct measurement of whether a webpage truly needs a content refresh.

Second, the data is observational. The model can identify measured relationships and useful prioritization signals, but it cannot establish that refreshing a page will cause rankings, clicks, or traffic to improve.

Third, the random development split allowed client overlap, which can make performance appear stronger than performance on unseen clients. The grouped validation is therefore more informative for assessing generalization.

Fourth, search performance can change because of factors that are not represented in the available features, including changes in search demand, competition, search-engine behavior, seasonality, and changes to the surrounding website.

Finally, the model should be treated as **decision-support**, not as an automated content-update system. A high score means that a page may deserve earlier review; it does not mean that an update is definitely required or that a particular change will improve performance.

## 6. Ranked recommendations

The model output should be used as a ranked review queue rather than an automatic publishing or editing system.

### 1. Review the highest-priority pages first

Pages receiving the highest model scores should be reviewed first because the model identifies them as stronger candidates for attention based on the measured historical signals.

### 2. Check recent search performance

Before making any content change, review recent impressions, clicks, and average position to confirm that the observed performance pattern is still relevant.

### 3. Check query visibility and concentration

Review the page's visible query coverage and query concentration. A page with limited or highly concentrated query visibility may require a different content strategy from a page with broader search coverage.

### 4. Consider content age

Older content can be prioritized for review when its other signals also indicate potential attention is warranted. Age alone should not trigger an automatic refresh.

### 5. Require human review before action

An SEO specialist or content manager should inspect the page, search intent, current content quality, and business context before deciding whether to update, consolidate, leave unchanged, or investigate further.

## 7. Artifacts the paper embeds

The deployed paper will use a small set of reproducible artifacts generated from this notebook.

The planned artifacts are:

1. **Model vs baseline results table** — showing the same metric and evaluation population.
2. **Validation comparison** — showing the development/random split alongside the grouped-by-client evaluation.
3. **Feature importance chart** — showing which measured features contributed most to the Random Forest model.
4. **Ranked action queue** — generated from held-out model predictions and exported for the action playbook.

Each artifact will be generated from the notebook so that the published paper can be traced back to the analysis.

In [ ]:
import os

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

print("Output directories ready:")
print("-", os.path.abspath("work/outputs"))
print("-", os.path.abspath("work/figures"))

Output directories ready:
- /content/work/outputs
- /content/work/figures


## Acknowledgments & Data Credit

Built on the FlyRank ML Internship dataset. The dataset was provided as part of the FlyRank ML Internship and was used for educational research and decision-support analysis.

Data source: [FlyRank](https://flyrank.ai)

# ML-12 — 5-Minute Demo Outline

## 1. The question — 45 seconds

My project focuses on the Refresh / Content Opportunity Scoring problem.

The main question is:

**Can historical search-performance signals help prioritize webpages that should be reviewed for a possible content refresh?**

The goal is not to automatically decide which pages must be changed, but to provide decision-support for SEO specialists and content managers.

## 2. The data and method — 1 minute

I worked with the FlyRank ML Internship dataset and used webpage-level search-performance information.

The final model used features that were available before the outcome window, including:

- Previous 30-day impressions
- Previous 30-day clicks
- Previous 30-day average position
- Visible query count
- Top query share
- Content age

I deliberately excluded outcome-derived fields such as `trend_direction`, `trend_pct`, and the declining label from the model features.

## 3. One chart — 1 minute

The main chart I would show is the model-versus-baseline evaluation from the validation analysis.

The purpose of the chart is to show how the model performs relative to a simple baseline, rather than presenting the model score by itself.

The key point is that the model should be interpreted as a ranking and decision-support tool, not as proof that a page will decline or that refreshing it will improve performance.

## 4. One honest result — 1 minute

Under the random split, the model achieved:

- Average Precision: **0.3874**
- ROC-AUC: **0.7436**
- Test base rate: **19.98%**

The random split also had substantial client overlap between training and test data, so I did not treat this result as the final evidence of generalization.

I therefore used an honest grouped validation approach by client and interpreted the difference between the random and grouped results as an important part of the analysis.

## 5. One recommendation — 1 minute

The model output can be used to create a ranked review queue.

SEO teams can start with the highest-priority pages and inspect the reasons behind their ranking, such as:

- declining recent search signals,
- low recent clicks,
- limited query visibility,
- or older content.

A human should review the page before any content changes are made.

The model should support prioritization, not automatically publish, rewrite, delete, or update content.

## Closing — 15 seconds

The main takeaway is that machine learning can provide a useful directional signal for prioritizing content reviews, but the value of the system depends on honest validation, leakage checks, and human review.

## Social Post

I completed my Machine Learning capstone with FlyRank, focusing on whether historical content-performance signals can help prioritize content for potential refresh or review.

I evaluated a Random Forest model using historical performance, search visibility, and content-age features, comparing row-random validation with client-grouped validation to test the robustness of the observed signal.

The client-grouped evaluation measured an Average Precision of 0.4131 against a 23.87% positive-rate baseline. The output is intended as decision-support for human review, not automated content decisions.


## Employer-Facing Summary

I built a machine-learning decision-support workflow to rank content that may deserve review for potential decline.

The analysis used pseudonymized FlyRank content-performance data with historical performance, search visibility, and content-age features, and evaluated the model using both random and client-grouped validation.

The client-grouped evaluation measured an Average Precision of 0.4131 against a 23.87% positive-rate baseline, providing directional evidence that the model can support content-review prioritization.

## Self-check

Before you submit, confirm each line honestly:

☑️ Every section above is filled — markdown thinking and supporting code

☑️ The notebook runs top to bottom with no errors (Runtime → Run all)

☑️ No client names, URLs, or private queries anywhere

☑️ Claims use careful language: observed, measured, directional, decision-support

☑️ Committed to your repo under work/notebooks/

☑️ Deployed paper has all 9 required sections, including Abstract and Acknowledgments & Data Credit with the FlyRank link

☑️ ML-12 completed in the notebook's closing cells: 5-minute demo outline + social-post cut + 3-sentence employer-facing summary
